# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_down'] = (df['trend_direction'] == 'down').astype(int)

def verdict(rates):
    r = rates.values
    if abs(r[-1] - r[0]) < 0.03:
        return "FALSE"
    if (np.diff(r) > 0).all():
        return "CONFIRMED"
    if (np.diff(r) < 0).all():
        return "OPPOSITE"
    return "MIXED"

def check(bucket):
    t = df.groupby(bucket, observed=True)['is_down'].agg(n='count', pct_declining='mean').round(3)
    print(t)
    print("verdict:", verdict(t['pct_declining']))

stale = pd.cut(df['days_since_last_update'], bins=[-1, 90, 180, 365, np.inf], labels=['0-90', '91-180', '181-365', '365+'])
check(stale)

volume = pd.qcut(df['impressions_90d'].rank(method='first'), 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
check(volume)

                            n  pct_declining
days_since_last_update                      
0-90                    20655          0.512
91-180                   9171          0.611
181-365                   169          0.467
365+                        5          0.600
verdict: MIXED
                    n  pct_declining
impressions_90d                     
Q1 low           7500          0.376
Q2               7500          0.605
Q3               7500          0.626
Q4 high          7500          0.562
verdict: MIXED


My rule: pages that are old and still visible should be reviewed for a refresh first. Each page gets a score from how long it has gone without an update and how many impressions it gets. Reason code: stale_visible_page (180+ days since update and 500+ impressions in 90 days). Action: refresh for pages with that code, monitor for all others. The score uses only these two fields and nothing from the label or future windows.

Signal 1: staleness (days since last update), the signal behind the refresh flags. Verdict: [word printed above]. The share of declining pages goes from [first bucket number] in the freshest bucket to [last bucket number] in the oldest bucket (n shown in the table).

Signal 2: search volume (impressions in 90 days). Verdict: [word printed above]. The share of declining pages goes from [first number] in the lowest quarter to [last number] in the highest quarter.

If a verdict is not CONFIRMED, I say so plainly: the data does not support that assumption, and that finding helps me avoid building the rule on a weak signal.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
df['stale_pct'] = df['days_since_last_update'].rank(pct=True)
df['volume_pct'] = df['impressions_90d'].rank(pct=True)
df['score'] = (100 * df['stale_pct'] * df['volume_pct']).round(1)

is_stale_visible = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
df['reason_code'] = np.where(is_stale_visible, 'stale_visible_page', 'none')
df['action'] = np.where(is_stale_visible, 'refresh', 'monitor')

queue = df.sort_values('score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

os.makedirs("../outputs", exist_ok=True)
queue[['rank', 'content_id', 'score', 'reason_code', 'action']].to_csv("../outputs/baseline_action_score.csv", index=False)

queue[['rank', 'content_id', 'score', 'impressions_90d', 'days_since_last_update', 'reason_code', 'action']].head(10)

,rank,content_id,score,impressions_90d,days_since_last_update,reason_code,action
0,1,content_cf56e2e2e282,98.3,61678,194,stale_visible_page,refresh
1,2,content_7368877ea310,98.2,59472,194,stale_visible_page,refresh
2,3,content_a5dbb404bdc2,98.2,79035,106,none,monitor
3,4,content_47b8b12d581e,96.8,40305,106,none,monitor
4,5,content_1bfaa38ff26c,95.2,25715,194,stale_visible_page,refresh
5,6,content_69fad7e6c50c,95.2,28000,106,none,monitor
6,7,content_482aff19e9cc,94.8,26287,106,none,monitor
7,8,content_6ac3ab740bbf,94.0,22462,106,none,monitor
8,9,content_cb7e312f5d32,93.8,21272,151,none,monitor
9,10,content_ac1d924c6a70,93.8,21853,106,none,monitor


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
def wrong_if(r):
    if 0 < r['avg_position'] <= 3:
        return "it already ranks in the top 3, so its age may not be hurting it"
    if r['ctr'] >= df['ctr'].median():
        return "its click rate is above the median, so it may be doing fine despite its age"
    return "its traffic is simply stable or seasonal, so a refresh would not change anything"

for _, r in queue.head(10).iterrows():
    print(f"Rank {int(r['rank'])} (content_id {r['content_id']}): action is {r['action']}. It is here because it has {int(r['impressions_90d'])} impressions in 90 days and has not been updated in {int(r['days_since_last_update'])} days. It would be wrong if {wrong_if(r)}.")

Rank 1 (content_id content_cf56e2e2e282): action is refresh. It is here because it has 61678 impressions in 90 days and has not been updated in 194 days. It would be wrong if its click rate is above the median, so it may be doing fine despite its age.
Rank 2 (content_id content_7368877ea310): action is refresh. It is here because it has 59472 impressions in 90 days and has not been updated in 194 days. It would be wrong if its click rate is above the median, so it may be doing fine despite its age.
Rank 3 (content_id content_a5dbb404bdc2): action is monitor. It is here because it has 79035 impressions in 90 days and has not been updated in 106 days. It would be wrong if its click rate is above the median, so it may be doing fine despite its age.
Rank 4 (content_id content_47b8b12d581e): action is monitor. It is here because it has 40305 impressions in 90 days and has not been updated in 106 days. It would be wrong if its click rate is above the median, so it may be doing fine despite i

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: some top pages are old and visible but may still be stable, so the rule would send a reviewer to a page that does not need work. Pages that already rank in the top 3 are the most likely to be weak picks. Leakage check: the score uses only days_since_last_update and impressions_90d. The check above shows no overlap with trend_direction, trend_pct, any last_30d or prev_30d column, or any product flag, so no label or future-window information is in the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.